<a href="https://colab.research.google.com/github/sl007ha/qqq-risk-monitor/blob/main/notebooks/03_real_cape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import requests
import io

# Shiller's monthly stock market data
# 数据从 1871 年开始，包含 P/E, CAPE, 实际利率等
SHILLER_URL = "https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/441f0d2c-37e4-4803-b4e2-8fe10407fbf6/ie_data.xls?ver=1778098504874"

response = requests.get(SHILLER_URL, timeout=30)
print(f"HTTP status: {response.status_code}")
print(f"File size: {len(response.content) / 1024:.1f} KB")

HTTP status: 200
File size: 1621.0 KB


In [3]:
# 把刚才的 response 内容读成 DataFrame
shiller_raw = pd.read_excel(io.BytesIO(response.content), sheet_name='Data', header=None)
print(shiller_raw.head(15))   # 看前 15 行
print(shiller_raw.tail(5))    # 看最后 5 行

                                                   0      1         2   \
0                                                 NaN    NaN       NaN   
1   Stock Market Data Used in "Irrational Exuberan...    NaN       NaN   
2                                  Robert J. Shiller     NaN       NaN   
3                                                 NaN    NaN       NaN   
4                                                 NaN    NaN       NaN   
5                                                 NaN    S&P       NaN   
6                                                 NaN  Comp.  Dividend   
7                                                Date      P         D   
8                                             1871.01   4.44      0.26   
9                                             1871.02    4.5      0.26   
10                                            1871.03   4.61      0.26   
11                                            1871.04   4.74      0.26   
12                                    

In [4]:
# 跳过前 7 行作为 header
shiller = pd.read_excel(io.BytesIO(response.content), sheet_name='Data', skiprows=7)
print(shiller.columns.tolist())
print(shiller.head())

['Date', 'P', 'D', 'E', 'CPI', 'Fraction', 'Rate GS10', 'Price', 'Dividend', 'Price.1', 'Earnings', 'Earnings.1', 'CAPE', 'Unnamed: 13', 'TR CAPE', 'Unnamed: 15', 'Yield', 'Returns', 'Returns.1', 'Real Return', 'Real Return.1', 'Returns.2']
      Date     P     D    E        CPI     Fraction Rate GS10       Price  \
0  1871.01  4.44  0.26  0.4  12.464061  1871.041667      5.32  118.545708   
1  1871.02   4.5  0.26  0.4  12.844641  1871.125000  5.323333  116.587763   
2  1871.03  4.61  0.26  0.4  13.034972  1871.208333  5.326667  117.693713   
3  1871.04  4.74  0.26  0.4  12.559226  1871.291667      5.33  125.596602   
4  1871.05  4.86  0.26  0.4  12.273812  1871.375000  5.333333  131.770822   

   Dividend     Price.1  ...  CAPE  Unnamed: 13  TR CAPE  Unnamed: 15  Yield  \
0  6.941866  118.545708  ...   NaN          NaN      NaN          NaN    NaN   
1  6.736182  117.149112  ...   NaN          NaN      NaN          NaN    NaN   
2  6.637823  118.816202  ...   NaN          NaN      NaN

In [5]:
# 找到 Date 列的真名（可能是 'Date' 也可能是别的）
print([c for c in shiller.columns if 'date' in str(c).lower() or 'Date' in str(c)])

# 假设叫 'Date'
shiller = shiller.dropna(subset=['Date'])  # 去掉表尾的注脚
shiller = shiller[shiller['Date'].apply(lambda x: isinstance(x, (int, float)))]

# 把 1871.01 这种数字转成日期
def shiller_date_to_timestamp(d):
    year = int(d)
    # 1871.01 → 月份是 0.01 那部分 × 100 = 1
    # 但浮点数有精度问题，所以用 round
    month = round((d - year) * 100)
    if month == 0:
        month = 1  # 偶尔出现的边界情况
    return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)

shiller['date'] = shiller['Date'].apply(shiller_date_to_timestamp)
shiller = shiller.set_index('date')
print(shiller.tail())

['Date']
               Date            P      D           E      CPI     Fraction  \
date                                                                        
2026-01-31  2026.01     6929.122  79.67  261.106667  325.252  2026.041667   
2026-02-28  2026.02  6893.806316  79.82  269.103333  326.785  2026.125000   
2026-03-31  2026.03  6654.419091  79.97  277.100000  330.213  2026.208333   
2026-04-30  2026.04  6957.007619    NaN         NaN  331.927  2026.291667   
2026-05-31  2026.05      7259.22    NaN         NaN  332.784  2026.375000   

           Rate GS10        Price   Dividend       Price.1  ...       CAPE  \
date                                                        ...              
2026-01-31      4.21  7089.582649  81.514952  4.813124e+06  ...  39.593576   
2026-02-28      4.13  7020.360301  81.285306  4.770727e+06  ...  38.943010   
2026-03-31      4.25  6706.229624  80.592637  4.561822e+06  ...  36.939759   
2026-04-30      4.32  6974.969868        NaN  4.744629e+06  .

In [6]:
# 找 CAPE 列
print([c for c in shiller.columns if 'CAPE' in str(c).upper()])
# 通常会有 'CAPE' 和 'TR CAPE'（总收益版本），我们用前者

cape = shiller[['CAPE']].copy()
cape.columns = ['cape']
cape = cape.dropna()
print(cape.tail())
print(f"CAPE 数据范围：{cape.index.min()} 到 {cape.index.max()}")
print(f"行数：{len(cape)}")

['CAPE', 'TR CAPE']
                 cape
date                 
2026-01-31  39.593576
2026-02-28  38.943010
2026-03-31  36.939759
2026-04-30  38.142618
2026-05-31  39.583506
CAPE 数据范围：1881-01-31 00:00:00 到 2026-05-31 00:00:00
行数：1745


In [7]:
window = 240  # 20 年 × 12 月
cape['cape_mean_20y'] = cape['cape'].rolling(window, min_periods=120).mean()
cape['cape_std_20y'] = cape['cape'].rolling(window, min_periods=120).std()
cape['cape_z'] = (cape['cape'] - cape['cape_mean_20y']) / cape['cape_std_20y']

# 看几个关键时刻
print("2000 dot-com 顶点前后:")
print(cape.loc['1999-06':'2000-06', ['cape', 'cape_z']])

print("\n2007 GFC 之前:")
print(cape.loc['2007-01':'2007-12', ['cape', 'cape_z']])

print("\n最近:")
print(cape.tail(6)[['cape', 'cape_z']])

2000 dot-com 顶点前后:
                 cape    cape_z
date                           
1999-06-30  42.180676  2.881219
1999-07-31  43.828036  3.004878
1999-08-31  41.930712  2.732766
1999-09-30  41.323451  2.615715
1999-10-31  40.552854  2.487309
1999-11-30  43.208291  2.728305
1999-12-31  44.197940  2.780427
2000-01-31  43.772578  2.683438
2000-02-29  42.185636  2.472980
2000-03-31  43.220748  2.538107
2000-04-30  43.528574  2.527082
2000-05-31  41.966051  2.330920
2000-06-30  42.781972  2.376757

2007 GFC 之前:
                 cape    cape_z
date                           
2007-01-31  27.207537  0.255823
2007-02-28  27.315181  0.264011
2007-03-31  26.227606  0.122883
2007-04-30  26.976268  0.211873
2007-05-31  27.548490  0.278721
2007-06-30  27.418263  0.257286
2007-07-31  27.410088  0.251440
2007-08-31  26.148607  0.087401
2007-09-30  26.725743  0.156271
2007-10-31  27.320648  0.226518
2007-11-30  25.729054  0.016786
2007-12-31  25.955510  0.039343

最近:
                 cape    cape_z
da

In [8]:
cape[['cape', 'cape_z']].to_csv('/content/cape.csv')